In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
data = joblib.load("../data/processed/model_ready_v1.pkl")
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]
train_meta, test_meta = data["train_meta"], data["test_meta"]

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

models = {
    "LinearRegression (baseline)": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1),
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds),
    })
    fitted_models[name] = model

results_df = pd.DataFrame(results).sort_values("MAE")
print(results_df)

                         model        MAE       RMSE        R2
2                      XGBoost  33.188370  60.704892  0.121696
1                 RandomForest  35.398129  63.541761  0.037688
0  LinearRegression (baseline)  35.518174  62.788036  0.060382


### Hyperparameter Tuning

In [4]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.85, 1.0],
}

tscv = TimeSeriesSplit(n_splits=3)

search = RandomizedSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV MAE:", -search.best_score_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best params: {'subsample': 0.7, 'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.05}
Best CV MAE: 24.326849406096937


In [5]:
best_model = search.best_estimator_
preds = best_model.predict(X_test)

tuned_mae = mean_absolute_error(y_test, preds)
tuned_rmse = np.sqrt(mean_squared_error(y_test, preds))
tuned_r2 = r2_score(y_test, preds)

print(f"Tuned model — MAE: {tuned_mae:.2f} min, RMSE: {tuned_rmse:.2f} min, R2: {tuned_r2:.3f}")
print(f"Improvement over Day 3 default: {results_df.iloc[0]['MAE'] - tuned_mae:.2f} min")

Tuned model — MAE: 32.72 min, RMSE: 60.29 min, R2: 0.134
Improvement over Day 3 default: 0.47 min


In [6]:
eval_df = test_meta.copy()
eval_df["actual"] = y_test.values
eval_df["predicted"] = preds
eval_df["abs_error"] = (eval_df["actual"] - eval_df["predicted"]).abs()

print("By station_code (top 10 worst):")
print(eval_df.groupby("station_code")["abs_error"].mean().sort_values(ascending=False).head(10))

print("\nBy month:")
print(eval_df.groupby("month")["abs_error"].mean())

print("\nBy rain_category:")
print(eval_df.groupby("rain_category")["abs_error"].mean())

By station_code (top 10 worst):
station_code
PERN    53.456711
KRMI    48.016286
THVM    42.435357
PNVL    40.110844
CSMT    40.086209
RN      39.152196
CHI     38.796476
KUDL    38.498985
SGR     37.698338
KYN     37.304802
Name: abs_error, dtype: float64

By month:
month
1     33.826948
2     45.181735
12    24.643086
Name: abs_error, dtype: float64

By rain_category:
rain_category
No rain    31.271365
Unknown    33.255586
Name: abs_error, dtype: float64


In [7]:
from pathlib import Path

Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, "../models/delay_model_v1.pkl")
joblib.dump(list(X_train.columns), "../models/feature_columns_v1.pkl")
print("Saved to ../models/")

Saved to ../models/
